# JSON - Structure

Introduce a missing measurement key, a misaligned hourly array, or an incorrect unit declaration.

**Instructions:** Use `prepare_data_json.ipynb` to create a clean `data/weather.json`. This notebook overwrites it; restore clean data before another run. These defects intentionally run **last** and may prevent ordinary normalization. The analyst needs an expected field/unit contract to establish a mismatch.

In [ ]:
from pathlib import Path
from random import Random
import copy
import json

import pandas as pd

# Configuration
DATA_PATH = Path("data/weather.json")
TIME_COLUMN = "time"
TIMEZONE = "America/Edmonton"
START_DATE = "2025-08-01"
END_DATE = "2025-09-01"  # Exclusive, local calendar dates.
SEED = 814205

# JSON Structure
STRUCTURE_KIND = "random"  # "missing_key", "misaligned", "unit_mismatch", "random".
STRUCTURE_TARGET_COLUMN = "relative_humidity_2m"
STRUCTURE_REPLACEMENT_UNIT = "fraction"  # Deliberately incorrect for source humidity percentages.


In [ ]:
def add_json_structure_error(
    document, rng, *, target_column, kind, replacement_unit,
    time_column, timezone, start_date, end_date,
):
    if kind not in ("missing_key", "misaligned", "unit_mismatch", "random"):
        raise ValueError("Unknown JSON structure error.")
    chosen = rng.choice(("missing_key", "misaligned", "unit_mismatch")) if kind == "random" else kind
    hourly = document.get("hourly")
    if not isinstance(hourly, dict) or target_column == time_column or target_column not in hourly:
        raise ValueError("Expected an existing measurement array.")
    if not all(isinstance(values, list) for values in hourly.values()):
        raise ValueError("Expected list-valued hourly fields.")
    if len({len(values) for values in hourly.values()}) != 1:
        raise ValueError("Hourly arrays are already misaligned. Restore clean data first.")

    dirty = copy.deepcopy(document)
    if chosen == "missing_key":
        del dirty["hourly"][target_column]
    elif chosen == "unit_mismatch":
        units = dirty.get("hourly_units")
        if not isinstance(units, dict) or target_column not in units:
            raise ValueError("Target has no existing unit declaration.")
        if units[target_column] == replacement_unit:
            raise ValueError("Replacement unit must differ from the existing unit.")
        units[target_column] = replacement_unit
    else:
        times = hourly.get(time_column)
        if not isinstance(times, list) or not times:
            raise ValueError("Expected an hourly time array.")
        start = pd.Timestamp(start_date, tz=timezone).tz_convert("UTC").timestamp()
        end = pd.Timestamp(end_date, tz=timezone).tz_convert("UTC").timestamp()
        if start >= end:
            raise ValueError("Start date must precede end date.")
        eligible = [i for i, value in enumerate(times)
                    if isinstance(value, (int, float)) and start <= value < end]
        if not eligible:
            raise ValueError("No observations in the selected date range.")
        index = rng.choice(eligible)
        del dirty["hourly"][target_column][index]
    return dirty


In [ ]:
# Small, reproducible source-shaped trial (no network required).
times = pd.date_range("2025-08-01", periods=24*14, freq="h", tz=TIMEZONE).tz_convert("UTC")
trial = {
    "latitude": 51.04, "timezone": "GMT", "hourly_units": {"temperature_2m": "°C", "relative_humidity_2m": "%"},
    "hourly": {
        "time": [int(t.timestamp()) for t in times],
        "temperature_2m": [float(i % 24) for i in range(len(times))],
        "relative_humidity_2m": [float(40 + i % 50) for i in range(len(times))],
    },
}

for mechanism in ("missing_key", "misaligned", "unit_mismatch"):
    candidate = add_json_structure_error(
        trial, Random(SEED), target_column=STRUCTURE_TARGET_COLUMN,
        kind=mechanism, replacement_unit=STRUCTURE_REPLACEMENT_UNIT,
        time_column=TIME_COLUMN, timezone=TIMEZONE,
        start_date=START_DATE, end_date=END_DATE,
    )
    json.dumps(candidate, allow_nan=False)
    assert candidate != trial
    print(mechanism, {name: len(v) for name, v in candidate["hourly"].items()},
          candidate.get("hourly_units", {}).get(STRUCTURE_TARGET_COLUMN))


In [ ]:
# Apply to the prepared JSON; the test fixture is used if it is absent.
if DATA_PATH.exists():
    clean = json.loads(DATA_PATH.read_text(encoding="utf-8"))
else:
    clean = trial
    print("Prepared data/weather.json not found: showing controlled trial only.")

dirty = add_json_structure_error(
    clean, Random(SEED), target_column=STRUCTURE_TARGET_COLUMN,
    kind=STRUCTURE_KIND, replacement_unit=STRUCTURE_REPLACEMENT_UNIT,
    time_column=TIME_COLUMN, timezone=TIMEZONE,
    start_date=START_DATE, end_date=END_DATE,
)
serialized = json.dumps(dirty, allow_nan=False)
if DATA_PATH.exists():
    DATA_PATH.write_text(serialized, encoding="utf-8")
saved = json.loads(serialized)


In [ ]:
# Inspect structure before attempting DataFrame construction.
print("Expected field present:", STRUCTURE_TARGET_COLUMN in saved["hourly"])
print("Units:", clean["hourly_units"].get(STRUCTURE_TARGET_COLUMN),
      "→", saved["hourly_units"].get(STRUCTURE_TARGET_COLUMN))
print("Hourly array lengths:", {field: len(values) for field, values in saved["hourly"].items()})
lengths = pd.DataFrame({
    "field": list(clean["hourly"]),
    "clean_length": [len(v) for v in clean["hourly"].values()],
    "dirty_length": [len(saved["hourly"][k]) if k in saved["hourly"] else None for k in clean["hourly"]],
})
lengths

In [ ]:
# Verify the chosen structural change is the only intended change.
assert saved != clean
assert json.loads(json.dumps(saved, allow_nan=False)) == saved
assert {k:v for k,v in saved.items() if k not in ("hourly", "hourly_units")} == {k:v for k,v in clean.items() if k not in ("hourly", "hourly_units")}
if STRUCTURE_TARGET_COLUMN not in saved["hourly"]:
    assert set(clean["hourly"]) - set(saved["hourly"]) == {STRUCTURE_TARGET_COLUMN}
    assert all(saved["hourly"][k] == clean["hourly"][k] for k in saved["hourly"])
elif saved["hourly_units"].get(STRUCTURE_TARGET_COLUMN) != clean["hourly_units"].get(STRUCTURE_TARGET_COLUMN):
    assert saved["hourly"] == clean["hourly"]
else:
    assert len(saved["hourly"][STRUCTURE_TARGET_COLUMN]) == len(clean["hourly"][STRUCTURE_TARGET_COLUMN]) - 1
    assert all(saved["hourly"][k] == clean["hourly"][k] for k in clean["hourly"] if k != STRUCTURE_TARGET_COLUMN)
